# StormVerify-Py: NWS Warning Verification & Spatial Audit

**Event Date:** April 27, 2026  
**Warning Types:** Tornado Warnings (TOR) and Severe Thunderstorm Warnings (SVR)  
**Data Source:** Iowa Environmental Mesonet (IEM)

This notebook walks through a spatial pipeline that evaluates whether NWS convective warnings were verified by ground-truth Local Storm Reports (LSRs). A warning is verified only if a qualifying report occurs **inside the polygon** while the warning is **still active**.

---

### Verification Rules

| Warning Type | Verified By |
|---|---|
| Tornado Warning (TOR) | Tornado reports only |
| Severe Thunderstorm Warning (SVR) | Hail OR damaging wind reports |

> **Note:** Storm surveys for April 27 are ongoing. Tornado report counts may increase as surveys conclude.

## Step 1: Imports and Environment Setup

Import libraries and set up the directory structure for raw and processed data. Ensure `data/raw` and `data/processed` folders exist before moving on.

In [1]:
# Import Libraries
import requests
import duckdb
import pandas as pd
import geopandas as gpd
import os
from shapely import wkb
from pathlib import Path

# Create folders for raw and processed data.
Path("data2").mkdir(exist_ok=True)
Path("data2/processed").mkdir(parents=True, exist_ok=True)
Path("data2/raw").mkdir(parents=True, exist_ok=True)
raw_folder = os.path.join("data", "raw")
processed_folder = os.path.join("data", "processed")

## Step 2: Ingest Data from IEM

Pull two datasets from the IEM API for the April 27, 2026 24-hour window:

- **LSRs** — Local Storm Reports (point data, ground-truth observations)
- **Storm-Based Warnings** — polygon warnings issued by NWS WFOs

Use the **Storm Based Warnings** option in the IEM warning archive, as this gives actual drawn polygons rather than county outlines, which matters a lot for spatial accuracy.

Save both files as GeoJSON to `data/raw/` directory.

For constructing parameters for IEM requests, see URLs below:

LSRs: https://mesonet.agron.iastate.edu/cgi-bin/request/gis/lsr.py?help

Warnings: https://mesonet.agron.iastate.edu/cgi-bin/request/gis/watchwarn.py?help

In [2]:
# Define parameters for appending to the URL. Documentation can be found on IEM site (See links above)
param_lsr = {'magge':'','wfo':'ALL','state':'_ALL','year1':2026, 'month1':4,'day1':27,'hour1':12,'minute1':00,'year2':2026, 'month2':4,'day2':28,'hour2':11,'minute2':59, 'type':['TORNADO','HAIL','TSTM WND DMG','TSTM WND GST'] , 'fmt':'csv'}
param_warn = {'accept':'shapefile', 'wfo':'ALL', 'limit1':True,'year1':2026, 'month1':4,'day1':27,'hour1':12,'minute1':00,'year2':2026, 'month2':4,'day2':28,'hour2':11,'minute2':59}

# Grab LSR CSV
with requests.get('https://mesonet.agron.iastate.edu/cgi-bin/request/gis/lsr.py', params=param_lsr, stream=True) as response:
    response.raise_for_status()

    lsr_filename = os.path.join('data','raw','lsr_iem.csv') 

    with open(lsr_filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print("LSR CSV Downloaded!")

# Grab warning zipped shapefile
with requests.get('https://mesonet.agron.iastate.edu/cgi-bin/request/gis/watchwarn.py', params=param_warn, stream=True) as response:
    response.raise_for_status()

    warnings_filename = os.path.join('data', 'raw', 'warnings.zip')

    with open(warnings_filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print("Storm Warning Shapefile Downloaded!")


LSR CSV Downloaded!
Storm Warning Shapefile Downloaded!


## Step 3: Preprocess and Filter

Load the raw GeoJSON files and filter both datasets down to what is actually needed.

**Warnings:** keeping only TOR and SVR. Drop everything else.

**LSRs:** keeping only report types that can verify one of your two warning types — tornado, hail, large hail, wind, and high wind reports.

In [3]:
# Load into geopandas
gdf_lsr = gpd.read_file(lsr_filename) # LSR CSV
gdf_warn = gpd.read_file(r'zip://data/raw/warnings.zip') # Storm Warnings shapefile

# Field type conversions
gdf_lsr['VALID'] = pd.to_datetime(gdf_lsr['VALID']) # convert string date/time field (VALID) to datetime
gdf_warn['ISSUED'] = pd.to_datetime(gdf_warn['ISSUED']) # convert string date/time field (ISSUED ) to datetime to match VALID
gdf_warn['EXPIRED'] = pd.to_datetime(gdf_warn['EXPIRED'])# convert string date/time field (EXPIRED) to datetime to match VALID
gdf_warn['INIT_ISS'] = pd.to_datetime(gdf_warn['INIT_ISS'])
gdf_warn['INIT_EXP'] = pd.to_datetime(gdf_warn['INIT_EXP'])
gdf_lsr['LAT'] = pd.to_numeric(gdf_lsr['LAT'],downcast='float') # Convert lat/lon fields from string to float
gdf_lsr['LON'] = pd.to_numeric(gdf_lsr['LON'],downcast='float')

# 'Dropping' LSR event with erroneous LAT value of 8.9 (being in Jackson, MO, which doesn't make sense geographically)
gdf_lsr = gdf_lsr[gdf_lsr['LAT'] > 9]

# 'Dropping' warnings except TOR and SVR
gdf_warn = gdf_warn.query("PHENOM in ['TO', 'SV']")

# Build point geometries in LSR gdf
gdf_lsr = gpd.GeoDataFrame(gdf_lsr,geometry=gpd.points_from_xy(gdf_lsr.LON, gdf_lsr.LAT),crs="EPSG:4326")

## Step 4: Convert to GeoParquet

DuckDB can read GeoJSON directly, but GeoParquet is much faster for spatial joins. Write both filtered datasets to `data/processed/` as Parquet files.

GeoPandas handles this natively with `.to_parquet()` — just make sure `pyarrow` is installed.

In [4]:
lsr_parquet = os.path.join("data", "processed", "lsr.parquet")
warn_parquet = os.path.join("data","processed", "warn.parquet")

# Convert geodataframes to parquet files
gdf_lsr.to_parquet(lsr_parquet)
gdf_warn.to_parquet(warn_parquet)

## Step 5: Initialize DuckDB and Load Spatial Extension

Open DuckDB connection and install and load the spatial extension. DuckDB reads Parquet files directly via `read_parquet()` in SQL

Run a quick row count on both files to confirm they loaded as expected.

In [5]:
#Open DuckDB connection
con = duckdb.connect()

# Install and Load Spatial Connection
con.install_extension("spatial")
con.load_extension("spatial")

# Load data into in-memory duckdb tables
ddb_lsr = con.read_parquet(lsr_parquet)
ddb_warn = con.read_parquet(warn_parquet)

# Quick rowcounts
#print(con.sql('''SELECT COUNT(*) FROM ddb_lsr'''))
#print(con.sql('''SELECT COUNT(*) FROM ddb_warn'''))

In [ ]:
# First 5 records from each table
#print(con.sql('''SELECT * FROM ddb_lsr LIMIT 5'''))
#print(con.sql('''SELECT * FROM ddb_warn LIMIT 5'''))

## Step 6: Spatial Join with Temporal Constraint

This is the core of the pipeline. Join LSR points to warning polygons using two conditions at once:

1. **Spatial:** the report point must fall inside the warning polygon
2. **Temporal:** the report time must fall between the warning's issue time and expire time

Using a`LEFT JOIN` so warnings with zero matching reports still appear in results, as those are false alarms.

A single report can match more than one warning if it falls inside overlapping polygons that are both active at the same time. This is intentional and reflects real NWS operational practice.

In [6]:
# Perform Spatio-Temporal Join
lsr_warn_join = con.sql('''
    SELECT
        w.wfo,
        w.etn,
        w.issued AS WARN_ISSUED,
        w.expired AS WARN_EXPIRED,
        l.valid as LSR_TIME,
        l.typetext,
        w.phenom AS WARN_TYPE,
        w.geometry AS warn_geom,
        l.geometry AS lsr_geom
FROM ddb_warn w
LEFT JOIN ddb_lsr l 
        ON ST_Intersects(w.geometry, l.geometry) -- Spatial Condition ()
        AND l.valid >= w.issued  -- Temporal condition start
        AND l.valid <= w.expired  -- Temporal condition end
         ''')

#lsr_warn_join.show()

## Step 7: Aggregate Report Counts by Warning

Collapse the join output to one row per warning. Rather than a single total count, produce **separate counts by report type** — tornado reports, hail reports, and wind reports each get their own column.

This is what distinguishes between a Tornado Warning that had hail inside it versus one that had an actual tornado report. A single total count would lose that distinction.

Build this as a CTE that wraps the join from Step 6.

In [7]:
# Aggregate Report Counts
warn_lsr_totals= con.sql('''SELECT wfo, etn, warn_type, warn_geom,
                        COUNT_IF(TYPETEXT = 'TORNADO') AS tornado_reports,
                        COUNT_IF(TYPETEXT = 'HAIL') AS hail_reports,
                        COUNT_IF(TYPETEXT = 'TSTM WND DMG' or TYPETEXT = 'TSTM WND GST') as wind_reports,
                        COUNT(TYPETEXT) as total_reports
                    FROM lsr_warn_join
                    GROUP BY wfo, etn, warn_type, warn_geom
                    ORDER BY total_reports DESC                     
                     ''')

#warn_lsr_totals.show()

## Step 8: Classify Warnings

With per-type report counts on each row, apply a `CASE WHEN` to assign a verification status label. The logic differs by warning type:

**TOR warnings:**
- Tornado reports present → Verified
- No tornado reports, but hail reports present → Unverified - Tornado w/ Hail
- No tornado reports, but wind reports present → Unverified - Tornado w/ Wind
- Nothing → Unverified - No Reports

**SVR warnings:**
- Hail OR wind reports present → Verified
- Tornado but no wind or hail → Unverified - SVR w/ Tornado Report
- Nothing → Unverified - No Reports

Also added a boolean `is_verified` column, used for the WFO stats calculation.


In [8]:
class_v_warns = con.sql('''
    SELECT wfo, etn, warn_type,
        CASE WHEN warn_type = 'TO' AND tornado_reports > 0 THEN 'Verified'
             WHEN warn_type = 'TO' AND (tornado_reports = 0 AND wind_reports > 0 AND hail_reports > 0) THEN 'Unverified - Tornado Warning w/ Hail + Wind'
             WHEN warn_type = 'TO' AND (tornado_reports = 0 AND hail_reports > 0) THEN 'Unverified - Tornado Warning w/ Hail'
             WHEN warn_type = 'TO' AND (tornado_reports = 0 AND wind_reports > 0) THEN 'Unverified - Tornado Warning w/ Wind'
             WHEN warn_type = 'TO' AND (tornado_reports = 0 AND wind_reports = 0 AND hail_reports = 0) THEN 'Unverified - No Reports'     
             WHEN warn_type = 'SV' AND (tornado_reports > 0 AND wind_reports = 0 AND hail_reports = 0) THEN 'Unverified - SVR w/ Tornado Report'     
             WHEN warn_type = 'SV' AND (wind_reports > 0 OR hail_reports > 0) THEN 'Verified'
             ELSE 'Unverified - No Reports'
             END AS verification_status,
        CASE WHEN verification_status LIKE 'Verified%' THEN True
             ELSE False
             END AS is_verified,
        tornado_reports, hail_reports, wind_reports, total_reports, warn_geom
    FROM warn_lsr_totals                       
''')

#class_v_warns.show()
#con.sql('''SELECT DISTINCT verification_status, COUNT(*), is_verified
        #FROM class_v_warns
        #GROUP BY verification_status, is_verified''')

## Step 9: Per-WFO Verification Stats

Roll up the classified warnings to the WFO level. Calculate total warnings issued, verified warnings, and verification rate per WFO and warning type.

Keep the raw counts alongside the rate — a WFO that issued 3 warnings tells a very different story than one that issued 40, even if their rates look similar.

In [9]:
# WFO verification stats
wfo_stats = con.sql('''SELECT wfo, warn_type, count(*) as total_warnings, SUM(is_verified::INT) AS verified_warnings, ROUND(AVG(is_verified::INT) * 100,2) AS verification_rate_pct
                    FROM class_v_warns
                    GROUP BY wfo, warn_type
                    ORDER BY warn_type, verification_rate_pct DESC''')

#wfo_stats.show()

## Step 10: Export Final GeoJSON

Pull the classified warnings into a GeoPandas GeoDataFrame and write to GeoJSON. Each feature should carry the verification status, is_verified flag, report counts, WFO, warning type, and timing columns.

Geometry will need to be reconstructed from the WKB column that came through the DuckDB pipeline before GeoPandas can write it out.

In [10]:
# Final warning table
final_warn = con.sql('''SELECT a.wfo, a.warn_type, a.verification_status, a.is_verified, a.tornado_reports, a.hail_reports, a.wind_reports, a.total_reports, b.issued, b.expired, a.warn_geom
                    FROM class_v_warns a
                    INNER JOIN ddb_warn b
                        ON a.wfo = b.wfo
                        AND a.etn = b.etn
                    GROUP BY a.wfo, a.warn_type, b.issued, b.expired, a.warn_geom, a.is_verified, a.verification_status, a.tornado_reports, a.hail_reports, a.wind_reports, a.total_reports
                    ''')

# Ensure geom column is converted to WKB blob prior to df
final_warn_df = con.execute("SELECT * EXCLUDE warn_geom, ST_asWKB(warn_geom) AS warn_geom FROM final_warn").df()

# Convert 'warn_geom' to Shapely geometries
final_warn_df['geometry'] = final_warn_df['warn_geom'].apply(lambda x: wkb.loads(bytes(x)))

# Create new GDF
final_warn_gdf = gpd.GeoDataFrame(final_warn_df, geometry='geometry', crs='EPSG:4326')

# Dropping original 'warn_geom' column
final_warn_gdf = final_warn_gdf.drop(columns=['warn_geom'])

# Export to GeoJSON
geojson_path = os.path.join("data", "processed", "verified_warnings.geojson")
final_warn_gdf.to_file(geojson_path, driver='GeoJSON')

# Export WFO stats to a CSV
csv_path = os.path.join("data", "processed", "wfo_stats.csv")
con.table("wfo_stats").write_csv(csv_path)

---

## Stretch Goals

- **Missed Event Analysis** — identify tornado reports that fell outside all active TOR warning polygons; check whether those reports fell inside SVR polygons instead (spin-up tornadoes in line convection)
- **Lead Time Calculation** — time between warning issuance and the first qualifying report inside the polygon
- **Date Range Script** — a command line script that prompts for a date range and runs the full pipeline automatically for any event period
